# L23 Intro to Generative AI (Part 2)

### Steps:
1. Set up import
2. Load the names file
3. Create the vocabulary (link numbers to letters)
4. Create training data (previous letter -> next letter)
5. Train a neural network (MLP in Scikit-Learn)
6. Generate 20 names using a function and a loop

### Set up imports

In [58]:
# !pip install scikit-learn

In [59]:
import random
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import numpy as np

### Load data and inspect it

In [60]:
with open('names.txt') as file:
    names = file.read().splitlines()

# only use the first 5000 most common names
names = names[:5000]
len(names)

5000

In [61]:
# look at the first 10 names
names[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

### Create links between numbers and letters (Vocab)

In [62]:
# get all the characters in the dataset
all_names = ''.join(names)

# get a list of the characters in alphabetical order
vocab = sorted(list(set(all_names)))

# add a space character for the start and end of a name
vocab = [' '] + vocab

len(vocab)

27

In [63]:
# create dictionaries to link numbers to letters
int_to_char = {index : val for index, val in enumerate(vocab)}
char_to_int = {val : index for index, val in enumerate(vocab)}

char_to_int

{' ': 0,
 'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26}

### Create training data

In [64]:
X = []
y = []

context_length = 7

# loop over the names
for name in names:
    # add a space character before and after each name
    name = ' '*context_length + name + ' '

    # loop over the letters in each name
    for index in range(len(name) - context_length):
        # get the context window
        context = name[index : index + context_length]

        # define the target letter
        target = name[index + context_length]

        # convert context and target to integers
        context_ints = [char_to_int[char] for char in context]
        target_int = char_to_int[target]

        # add data to the X and y lists
        X.append(context_ints)
        y.append(target_int)
len(y)


35245

### Train the NN model

In [65]:
# define our ML model
clf = MLPClassifier(random_state=42, hidden_layer_sizes=(100, 100, 100))

# train the model
clf.fit(X, y)

# print out the loss value (lower = better performace)
clf.loss_

/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


np.float64(1.8098046669943697)

### Look at example probability distribution

In [66]:
test_context = [0] * context_length

clf.predict_proba([test_context])

array([[5.94434148e-05, 1.55770119e-01, 4.87620963e-02, 4.86924864e-02,
        2.87865431e-02, 6.60849964e-02, 9.03040303e-03, 1.79548825e-02,
        2.28403756e-02, 2.39766125e-02, 6.00351244e-02, 8.78165639e-02,
        7.29709996e-02, 9.38370151e-02, 3.47957970e-02, 9.61784117e-03,
        1.26231481e-02, 1.87806583e-03, 4.84869299e-02, 7.07016058e-02,
        2.57169965e-02, 1.62492306e-03, 1.23946932e-02, 6.74496433e-03,
        2.41649648e-03, 1.23054206e-02, 2.40754564e-02]])

### Define a function for generating a new name

Steps: 
1. define a name variable that is an empty string
2. define the first character as a space (start of name indicator)
3. use a while loop (if the current character is not a space, then get the next letter)
4. convert the current letter to an integer
5. pass the integer into the model, and get probability distribution of next letter
6. select next letter using randomness with weighted probabilities
7. add the letter to the name
8. after the while loop ends (we get the end of name character), return the name

In [67]:
def generate_name():
    # define variable for the name
    name = ''

    # define initial values for current and next character
    context = ' ' * context_length
    next_char = ''

    # add characters until the next character is a space
    while next_char != ' ':
        # convert the context characters to integers
        context_ints = [char_to_int[char] for char in context]

        # generate the probability distribution of next character
        probs = clf.predict_proba([context_ints])[0]

        # select the next character using weighted probabilities
        next_char_int = np.random.choice(range(len(vocab)), p=probs)

        # convert the integer to a character
        next_char = int_to_char[next_char_int]

        # add the character to the name
        name += next_char

        # update the context
        context = context[1:] + next_char
    
    # Return the name

    return name

generate_name()

'denina '

### Activity
Write a loop to print 20 names and ensure they are all between 5 and 15 characters long

In [68]:
num_names = 0
while num_names != 20:
    name = generate_name()
    if len(name) >= 5 and len(name) <= 15:
        print(name)
        num_names += 1

keesier 
kaliah 
beelyne 
daria 
azanara 
caylynn 
eleanoe 
soliniy 
gracelynn 
malara 
kelsa 
jeali 
atiai 
marly 
avareia 
sabinh 
harhe 
isstin 
shernra 
lylah 


### Activity 2

Try changing context length to 5 or 7
try changing the NN hidden layers to (100, 100, 100) and analyze the loss. Is it lower than 2.11?

In [71]:
# changed context length to 5, changed NN hidden layers to (100, 100, 100). Loss value is lower than 2.11, now 1.83.
num_names = 0
while num_names != 20:
    name = generate_name()
    if len(name) >= 5 and len(name) <= 15:
        print(name)
        num_names += 1

vorelary 
addelyne 
areei 
lovaelee 
tatdnahd 
likli 
rylshae 
lamiyah 
barali 
dainyns 
daliy 
kabildt 
parilre 
zkoey 
maniya 
desha 
ancea 
iarmone 
adamlun 
azvha 
